# Photon Mosaic: synthetic data walkthrough

Generates a synthetic imaging movie with known ground-truth fluorescence, then runs it
through the full analysis pipeline: fluorescence extraction, neuropil correction, ΔF/F,
and deconvolution.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import spikeinterface.widgets as sw
from IPython.display import display
from ipywidgets import Dropdown, interactive_output

import photon_mosaic as pm
import photon_mosaic.widgets as pw

%matplotlib widget

## Generate synthetic imaging data

In [ ]:
rois, imaging, ground_truth = pm.generate_imaging_with_rois(
    num_frames=10000,
    bleaching_time=600.0,
    noise_std="poisson",
    weighted_rois=True,
    neuropil_model="diffuse",  # realistic, spatially-varying fluctuating background
    seed=0,
)

In [ ]:
rois

In [ ]:
imaging

In [ ]:
w = pw.plot_imaging_series(imaging, backend="ipywidgets", vmax_percentile=99.75)
w.colorbars["imaging"].set_label("Photon counts")
w.figure.tight_layout()  # re-run: the label wasn't there for the widget's own tight_layout() call

In [ ]:
pw.plot_rois(rois, backend="ipywidgets", width_cm=20)

## Create the ROI analyzer

In [ ]:
# Stays naive throughout; a second, corrected analyzer is introduced in "Neuropil
# correction" below.
analyzer_raw = pm.create_roi_analyzer(rois, imaging)

## Extract fluorescence

In [ ]:
fluorescence_ext = analyzer_raw.compute("fluorescence", n_jobs=4)

In [ ]:
fluorescence = fluorescence_ext.get_data(outputs="recording")

In [ ]:
sw.plot_traces(fluorescence, backend="ipywidgets", time_range=[0, 30])

## Neuropil correction

Subtracts each ROI's own Suite2p-style surround neuropil estimate from its raw fluorescence
trace above, removing most of the video's `background` term -- including its genuine,
spatially-varying fluctuation (many broad, overlapping, independently-fluctuating sources, so
nearby background pixels move together but distant ones don't), not just a constant offset.
From here on, `fluorescence_ext` refers to this corrected trace. The plot below shows what
this looks like for a single ROI; see "Effect of neuropil correction" further down for what
it buys us at the level of ground-truth recovery, and what happens if it's skipped.

In [ ]:
# A separate analyzer for the corrected pipeline: RoiAnalyzer keeps only one "fluorescence"
# extension at a time per analyzer (compute() overwrites it, cascading to delete
# dependents), so correcting directly on analyzer_raw would make the naive extraction above
# unavailable for the comparisons further down ("Quick look", "Effect of neuropil
# correction"). This analyzer becomes the default for the rest of the notebook.
analyzer = pm.create_roi_analyzer(rois, imaging)
analyzer.compute("neuropil", method="surround")
# neuropil_weight=1.0 (not the library's default 0.7): our diffuse background sources are
# uncorrelated with the cells by construction, so there's no risk of over-subtracting real
# correlated activity -- this gives a meaningfully larger ground-truth correlation gain.
fluorescence_ext = analyzer.compute("fluorescence", n_jobs=4, use_neuropil=True, neuropil_weight=1.0)

In [ ]:
fluorescence_raw = fluorescence.get_traces()  # naive snapshot from "Extract fluorescence" above, before correction
neuropil_signal = fluorescence_raw - fluorescence_ext.get_data()  # exact amount subtracted: F_raw - F_corrected

t = np.arange(fluorescence_ext.get_data().shape[0]) / imaging.sampling_frequency
fig_neuropil_trace, ax_neuropil_trace = plt.subplots(2, 1, figsize=(10, 2.5), sharex=True)
fig_neuropil_trace.supylabel("Fluorescence")  # shared: both rows are on the same (photon-count) scale


def _plot_neuropil_trace_roi(roi):
    for a in ax_neuropil_trace:
        a.clear()
    ax_neuropil_trace[0].plot(t, fluorescence_raw[:, roi], lw=0.5, label="F raw")
    ax_neuropil_trace[0].plot(t, neuropil_signal[:, roi], lw=0.5, label="neuropil")
    ax_neuropil_trace[0].legend(loc="upper right", fontsize=8)

    ax_neuropil_trace[1].plot(t, fluorescence_ext.get_data()[:, roi], c="C2", lw=0.5, label="F corrected")
    ax_neuropil_trace[1].set_xlabel("Time (s)")
    ax_neuropil_trace[1].legend(loc="upper right", fontsize=8)
    ax_neuropil_trace[-1].set_xlim(t[0], t[-1])
    fig_neuropil_trace.tight_layout()


# ROI 8: no mask overlap with neighbors, and a clean illustrative example of what
# correction buys us -- flat baseline with sharp transients once corrected. Chosen as a
# default for this walkthrough, not a representative case; switch ROIs below to see the range.
neuropil_roi_dropdown = Dropdown(options=[int(i) for i in rois.roi_ids], value=8, description="ROI:")
neuropil_trace_out = interactive_output(_plot_neuropil_trace_roi, {"roi": neuropil_roi_dropdown})
display(neuropil_roi_dropdown, neuropil_trace_out)

## Compute ΔF/F

In [ ]:
dff_ext = analyzer.compute("df_over_f", n_jobs=4, method="percentile")
df_over_f = dff_ext.get_data(outputs="recording")
sw.plot_traces(df_over_f, backend="ipywidgets", time_range=[30, 60])

### Quick look: does this help?

The "Neuropil correction" plot above shows what subtraction does to the raw trace; this
checks whether it actually helps recover the true underlying signal, using only what's
available at this point in the pipeline -- just to motivate the rest of the notebook.
"Effect of neuropil correction" at the very end repeats this same kind of comparison one
stage further, after denoising and deconvolution.

In [ ]:
# analyzer_raw (created in "Create the ROI analyzer", holding the naive fluorescence from
# "Extract fluorescence") just needs df_over_f added here for this comparison; reused again
# (with deconvolution added) in "Effect of neuropil correction" further down.
dff_ext_raw = analyzer_raw.compute("df_over_f", n_jobs=4, method="percentile")


def median_corr(dff, clean_traces):
    return np.median([np.corrcoef(dff[:, i], clean_traces[:, i])[0, 1] for i in range(clean_traces.shape[1])])


corr_wo_neuropil = median_corr(dff_ext_raw.get_data(), ground_truth.clean_traces)
corr_w_neuropil = median_corr(dff_ext.get_data(), ground_truth.clean_traces)
print("median correlation with ground truth without neuropil:", corr_wo_neuropil)
print("median correlation with ground truth with neuropil:   ", corr_w_neuropil)
print(f"change from subtraction: {corr_w_neuropil - corr_wo_neuropil:+.4f}")

fig_neuropil_quick, ax_neuropil_quick = plt.subplots(figsize=(10, 1.5))


def _plot_neuropil_quick_roi(roi):
    ax_neuropil_quick.clear()
    ax_neuropil_quick.plot(t, 100 * dff_ext_raw.get_data()[:, roi], c="C1", lw=0.5, alpha=0.7, label="without neuropil")
    ax_neuropil_quick.plot(t, 100 * dff_ext.get_data()[:, roi], c="C0", lw=0.5, alpha=0.7, label="with neuropil")
    ax_neuropil_quick.plot(t, 100 * ground_truth.clean_traces[:, roi], lw=0.5, ls="-.", c="k", label="ground truth")
    ax_neuropil_quick.set_ylabel(r"$\Delta F/F$ [%]")
    ax_neuropil_quick.set_xlabel("Time (s)")
    ax_neuropil_quick.legend(loc="upper right", fontsize=8)
    ax_neuropil_quick.set_xlim(t[0], t[-1])
    fig_neuropil_quick.tight_layout()


neuropil_quick_out = interactive_output(_plot_neuropil_quick_roi, {"roi": neuropil_roi_dropdown})
display(neuropil_roi_dropdown, neuropil_quick_out)

### Neuropil estimate vs. ground truth

The surround estimate above assumes nearby (non-ROI) pixels share each ROI's own local
background level. This synthetic data lets us check that assumption directly:
`ground_truth.neuropil` is the true, noise-free background under each ROI's own mask, while
the surround mask estimates it from nearby pixels instead.

In [ ]:
def fit_scale(inferred, ground_truth):
    return (inferred * ground_truth).sum(axis=0) / (inferred * inferred).sum(axis=0)


neuropil_masks_flat = analyzer.get_extension("neuropil").data["neuropil_masks"].reshape((rois.get_num_rois(), -1))
movie = imaging.get_series()
neuropil_estimate = movie.reshape(movie.shape[0], -1) @ neuropil_masks_flat.T

scale_neuropil_estimate = np.median(fit_scale(neuropil_estimate, ground_truth.neuropil))
corr_neuropil_estimate = median_corr(neuropil_estimate, ground_truth.neuropil)
print("median scale factor, surround estimate vs. true local neuropil (1.0 = unbiased):", scale_neuropil_estimate)
print("median correlation, surround estimate vs. true local neuropil:", corr_neuropil_estimate)

fig_neuropil_gt, ax_neuropil_gt = plt.subplots(figsize=(10, 1.5))


def _plot_neuropil_gt_roi(roi):
    ax_neuropil_gt.clear()
    ax_neuropil_gt.plot(t, neuropil_estimate[:, roi], lw=0.5, alpha=0.7, label="surround estimate")
    ax_neuropil_gt.plot(t, ground_truth.neuropil[:, roi], lw=0.5, ls="-.", c="k", label="true local neuropil")
    ax_neuropil_gt.set_xlim(t[0], t[-1])
    ax_neuropil_gt.set_ylabel("Neuropil")
    ax_neuropil_gt.set_xlabel("Time (s)")
    ax_neuropil_gt.legend(loc="upper right", fontsize=8)
    fig_neuropil_gt.tight_layout()


neuropil_gt_out = interactive_output(_plot_neuropil_gt_roi, {"roi": neuropil_roi_dropdown})
display(neuropil_roi_dropdown, neuropil_gt_out)

### Percentile vs. maximin baseline estimation

An aside comparing the two `df_over_f` baseline-estimation methods -- purely for
display below; deconvolution below uses the percentile method. Both are shown
here on the neuropil-corrected fluorescence established above.

In [ ]:
dff_ext_maximin = analyzer.compute("df_over_f", n_jobs=4, method="maximin")
# Restore dff_ext to the percentile baseline afterward: analyzer.compute("df_over_f", ...)
# overwrites whichever "df_over_f" extension is currently registered, and deconvolution
# below expects the percentile-based dff_ext, not this maximin comparison's.
dff_ext = analyzer.compute("df_over_f", n_jobs=4, method="percentile")

In [ ]:
t = np.arange(fluorescence_ext.get_data().shape[0]) / imaging.sampling_frequency
fig_methods, ax_methods = plt.subplots(2, 1, figsize=(10, 2.5), sharex=True)


def _plot_dff_methods(roi):
    for a in ax_methods:
        a.clear()
    ax_methods[0].plot(t, fluorescence_ext.get_data()[:, roi], c="gray", lw=0.5, label="F")
    ax_methods[0].plot(t, dff_ext.data["f0"][:, roi], c="C0", alpha=0.7, label=r"$F_0$ (percentile)")
    ax_methods[0].plot(t, dff_ext_maximin.data["f0"][:, roi], c="C1", ls="--", alpha=0.7, label=r"$F_0$ (maximin)")
    ax_methods[0].set_ylabel("Fluorescence")
    ax_methods[0].legend(loc="upper right", fontsize=8)

    dff1 = 100 * dff_ext.get_data()[:, roi]
    dff2 = 100 * dff_ext_maximin.get_data()[:, roi]
    ax_methods[1].axhline(0, ls="--", c="k")
    ax_methods[1].plot(t, dff1, c="C0", lw=0.5, alpha=0.7, label="percentile")
    ax_methods[1].plot(t, dff2, c="C1", lw=0.5, ls="--", alpha=0.7, label="maximin")
    ax_methods[1].set_xlim(t[0], t[-1])
    ax_methods[1].set_ylabel(r"$\Delta F/F$ [%]")
    ax_methods[1].set_xlabel("Time (s)")
    ax_methods[1].legend(loc="upper right", fontsize=8)
    fig_methods.tight_layout()


dff_method_dropdown = Dropdown(options=[int(i) for i in rois.roi_ids], value=8, description="ROI:")
dff_method_out = interactive_output(_plot_dff_methods, {"roi": dff_method_dropdown})
display(dff_method_dropdown, dff_method_out)

## Deconvolution

In [ ]:
deconv_ext = analyzer.compute("deconvolution", n_jobs=4)

In [ ]:
deconvolved = deconv_ext.get_data(outputs="recording")
sw.plot_traces(deconvolved, backend="ipywidgets", time_range=[30, 60])

### Reconstructed movies: raw, corrected, denoised

`traces @ ROIs` paints each ROI's trace value back onto its own patch of each frame via
the ROI's own mask -- the footprint takes on the mask's own spatial profile, scaled by
that value. `corrected` (same photon-count units as `raw`, using the neuropil-corrected
`fluorescence_ext` from above) isolates the effect of collapsing each ROI down to one
scalar per frame -- still temporally noisy. `FluorescenceNode`'s extraction (see its
docstring) is normalized to reconstruct this painting exactly for non-overlapping ROIs;
these ROIs do overlap (see the crosstalk check below), so it's only an approximation
here. `denoised` adds OASIS's temporal denoising on top; since OASIS runs on ΔF/F (not
raw F), its output is converted back to photon-count units via the fitted baseline
`dff_ext.data["f0"]` before painting, so all three panels stay directly comparable.

In [ ]:
raw_masks = rois.get_roi_image_masks()
masks_flat = raw_masks.reshape(rois.get_num_rois(), -1).astype(np.float32)
video_shape = imaging.get_series(epoch_index=0).shape
corrected_movie = (fluorescence_ext.get_data() @ masks_flat).reshape(video_shape)
# deconv_ext.data["denoised"] is on the dF/F scale (OASIS runs on dF/F, not raw F) --
# invert dF/F = (F - F0) / F0 to get back to photon counts before painting onto the movie.
denoised_F = deconv_ext.data["denoised"] * dff_ext.data["f0"] + dff_ext.data["f0"]
denoised_movie = (denoised_F @ masks_flat).reshape(video_shape)
corrected_imaging = pm.NumpyImaging(corrected_movie, sampling_frequency=imaging.sampling_frequency)
denoised_imaging = pm.NumpyImaging(denoised_movie, sampling_frequency=imaging.sampling_frequency)

w = pw.plot_imaging_series(
    {"Raw": imaging, "Corrected": corrected_imaging, "Denoised": denoised_imaging},
    backend="ipywidgets",
    width_cm=10,
    vmax_percentile=99.75,
)
for view in ("Raw", "Corrected", "Denoised"):
    w.colorbars[view].set_label("Photon counts")
w.figure.tight_layout()  # re-run: the labels weren't there for the widget's own tight_layout() call

# Corrected and Denoised are on the same (photon-count) scale, so share one contrast range for
# a fair visual comparison. Raw keeps its own (much wider) range: its shot noise spans most of
# that shared range, so the same clipping would saturate Raw to white and hide its structure.
shared_sample = np.concatenate([corrected_movie[:100].ravel(), denoised_movie[:100].ravel()])
shared_vmin, shared_vmax = np.percentile(shared_sample, [2.0, 99.75])
for view in ("Corrected", "Denoised"):
    w.global_vmin[view] = shared_vmin
    w.global_vmax[view] = shared_vmax
    w.images[view].set_clim(shared_vmin, shared_vmax)
    w.colorbars[view].update_normal(w.images[view])


### Denoised vs. ground truth (ΔF/F scale)

Validates recovery accuracy: `denoised` (native ΔF/F scale, no `F0` conversion) compared
directly against the true underlying signal, `ground_truth.clean_traces`. Painting both
through the same masks on the same ΔF/F scale makes the comparison direct.

In [ ]:
denoised_dff_movie = (100 * deconv_ext.data["denoised"] @ masks_flat).reshape(video_shape)
ground_truth_movie = (100 * ground_truth.clean_traces @ masks_flat).reshape(video_shape)
denoised_dff_imaging = pm.NumpyImaging(denoised_dff_movie, sampling_frequency=imaging.sampling_frequency)
ground_truth_imaging = pm.NumpyImaging(ground_truth_movie, sampling_frequency=imaging.sampling_frequency)

w2 = pw.plot_imaging_series(
    {"Denoised": denoised_dff_imaging, "Ground truth": ground_truth_imaging},
    backend="ipywidgets",
    width_cm=10,
    vmax_percentile=99.75,
)
for view in ("Denoised", "Ground truth"):
    w2.colorbars[view].set_label(r"$\Delta F/F$ [%]")
w2.figure.tight_layout()  # re-run: the labels weren't there for the widget's own tight_layout() call

# same units for both views -- share one contrast range for a fair comparison.
shared_sample2 = np.concatenate([denoised_dff_movie[:100].ravel(), ground_truth_movie[:100].ravel()])
shared_vmin2, shared_vmax2 = np.percentile(shared_sample2, [2.0, 99.75])
for view in ("Denoised", "Ground truth"):
    w2.global_vmin[view] = shared_vmin2
    w2.global_vmax[view] = shared_vmax2
    w2.images[view].set_clim(shared_vmin2, shared_vmax2)
    w2.colorbars[view].update_normal(w2.images[view])


**Note:** the cell below checks which of the plotted ROIs share pixels with other ROIs.
Unmixed crosstalk between overlapping ROIs can cause small false transients/events in the
traces shown further down -- try ROI 1 in the dropdown below (it overlaps with ROIs 7 and
14): an occasional small deconvolved "event" with no corresponding true spike is this effect.

In [ ]:
masks = rois.get_roi_image_masks() > 0
for i in rois.roi_ids[:3]:
    overlapping = [int(j) for j in rois.roi_ids if j != i and np.any(masks[i] & masks[j])]
    print(f"ROI {i} overlaps with: {overlapping}")

**Note:** the panels below use the neuropil-corrected pipeline established above (the
default). See "Effect of neuropil correction" further down for what recovery looks like,
and how much worse it gets, without that correction.

In [ ]:
t = np.arange(dff_ext.get_data().shape[0]) / imaging.sampling_frequency
fig_roi, ax_roi = plt.subplots(3, 1, figsize=(10, 4), sharex=True)


def _plot_roi(roi):
    for a in ax_roi:
        a.clear()
    ax_roi[0].plot(t, fluorescence_ext.get_data()[:, roi], lw=0.5, label="F corrected")
    ax_roi[0].plot(t, dff_ext.data["f0"][:, roi], c="#F0E442", label=r"$F_0$")
    ax_roi[0].set_ylabel("Fluorescence")
    ax_roi[0].legend(loc="upper right", fontsize=8)

    ax_roi[1].plot(t, 100 * dff_ext.get_data()[:, roi], c="gray", lw=0.5, alpha=0.7, label=r"inferred $\Delta F/F$")
    ax_roi[1].plot(t, 100 * deconv_ext.data["denoised"][:, roi], c="C0", lw=0.5, alpha=0.7, label="denoised")
    ax_roi[1].plot(t, 100 * ground_truth.clean_traces[:, roi], c="k", lw=0.5, ls="-.", label="ground truth")
    ax_roi[1].set_ylabel(r"$\Delta F/F$ [%]")
    ax_roi[1].legend(loc="upper right", fontsize=8)

    ax_roi[2].plot(t, deconv_ext.get_data()[:, roi], c="C0", lw=0.5, label="deconvolved")
    ax_roi[2].set_xlim(t[0], t[-1])
    spike_times = t[ground_truth.spikes[:, roi] > 0]
    ax_roi[2].vlines(
        spike_times, 0.9, 1.0, transform=ax_roi[2].get_xaxis_transform(), color="k", label="true spikes"
    )
    ax_roi[2].set_ylabel("Activity [a.u.]")
    ax_roi[2].set_xlabel("Time (s)")
    ax_roi[2].legend(loc="upper right", fontsize=8)
    fig_roi.tight_layout()


roi_dropdown = Dropdown(options=[int(i) for i in rois.roi_ids], value=8, description="ROI:")
out = interactive_output(_plot_roi, {"roi": roi_dropdown})
display(roi_dropdown, out)

## Effect of neuropil correction

The same before/after comparison as "Quick look: does this help?" above, one stage
further -- after denoising and deconvolution -- to tie the whole pipeline together as a
final summary of what the correction buys us.

In [ ]:
# Reuses analyzer_raw (fluorescence from "Extract fluorescence", df_over_f from "Quick
# look" above); only deconvolution is new here.
deconv_ext_raw = analyzer_raw.compute("deconvolution", n_jobs=4)

In [ ]:
corr_wo_neuropil_denoised = median_corr(deconv_ext_raw.data["denoised"], ground_truth.clean_traces)
corr_w_neuropil_denoised = median_corr(deconv_ext.data["denoised"], ground_truth.clean_traces)
print("median correlation with ground truth, denoised, without neuropil:", corr_wo_neuropil_denoised)
print("median correlation with ground truth, denoised, with neuropil:   ", corr_w_neuropil_denoised)
print(f"change from subtraction: {corr_w_neuropil_denoised - corr_wo_neuropil_denoised:+.4f}")

corr_wo_neuropil_spikes = median_corr(deconv_ext_raw.get_data(), ground_truth.spikes)
corr_w_neuropil_spikes = median_corr(deconv_ext.get_data(), ground_truth.spikes)
print("median correlation with true spikes, deconvolved, without neuropil:", corr_wo_neuropil_spikes)
print("median correlation with true spikes, deconvolved, with neuropil:   ", corr_w_neuropil_spikes)
print(f"change from subtraction: {corr_w_neuropil_spikes - corr_wo_neuropil_spikes:+.4f}")

fig_neuropil, ax_neuropil = plt.subplots(2, 1, figsize=(10, 2.5), sharex=True)


def _plot_neuropil_roi(roi):
    for a in ax_neuropil:
        a.clear()

    ax_neuropil[0].plot(t, 100 * deconv_ext_raw.data["denoised"][:, roi], c="C1", lw=0.5, alpha=0.7, label="without neuropil")
    ax_neuropil[0].plot(t, 100 * deconv_ext.data["denoised"][:, roi], c="C0", lw=0.5, alpha=0.7, label="with neuropil")
    ax_neuropil[0].plot(t, 100 * ground_truth.clean_traces[:, roi], lw=0.5, ls="-.", c="k", label="ground truth")
    ax_neuropil[0].set_ylabel("denoised\n" r"$\Delta F/F$ [%]", fontsize=8)
    ax_neuropil[0].legend(loc="upper right", fontsize=8)

    ax_neuropil[1].plot(t, deconv_ext_raw.get_data()[:, roi], c="C1", lw=0.5, alpha=0.7, label="without neuropil")
    ax_neuropil[1].plot(t, deconv_ext.get_data()[:, roi], c="C0", lw=0.5, alpha=0.7, label="with neuropil")
    spike_times = t[ground_truth.spikes[:, roi] > 0]
    ax_neuropil[1].vlines(
        spike_times, 0.9, 1.0, transform=ax_neuropil[1].get_xaxis_transform(), color="k", label="true spikes"
    )
    ax_neuropil[1].set_ylabel("Activity [a.u.]")
    ax_neuropil[1].set_xlabel("Time (s)")
    ax_neuropil[1].legend(loc="upper right", fontsize=8)

    ax_neuropil[-1].set_xlim(t[0], t[-1])
    fig_neuropil.tight_layout()


neuropil_roi_out = interactive_output(_plot_neuropil_roi, {"roi": neuropil_roi_dropdown})
display(neuropil_roi_dropdown, neuropil_roi_out)